In [ ]:
# import toml
# from pathlib import Path
# config_path = Path("../../../../configuration/asim_configuration")
# input_config = toml.load(config_path/ "input_configuration.toml")
# summary_config = toml.load(config_path/ "summary_configuration.toml")

The stop frequency model assigns to each tour the number of intermediate destinations a person will travel to on each leg of the tour from the origin to tour primary destination and back. Intermediate stops are not modeled for drive-transit tours

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import os, sys

module_path = os.path.abspath(os.path.join('../../../..')) # or the path to your source code
sys.path.insert(0, module_path)
import util

In [3]:
# read data
person = util.get_person_data(summary_config, uncloned=True)
hh_data = util.get_hh_data(summary_config, uncloned=True)
tour = util.get_tour_data(summary_config)

In [5]:
per_data = person.merge(hh_data[['household_id','auto_ownership','auto_ownership_4+','log_hh_1','log_hh_1_group','source']],
                          how='left', on=['household_id','source']) # get auto ownership from hh data

tour_data = tour.merge(per_data, how='left', on=['person_id','household_id','source'])

In [6]:
#| layout-ncol: 2
df_plot = tour_data.groupby(['source','stop_frequency_out'])['tour_weight'].sum().reset_index()
df_plot['percentage'] = df_plot.groupby(['source'], group_keys=False)['tour_weight']. \
    apply(lambda x: x / float(x.sum()))

fig = px.bar(df_plot, x="stop_frequency_out", y="percentage", color="source",barmode="group",
             title="outbound stop frequency")
fig.for_each_annotation(lambda a: a.update(text = a.text.split("=")[-1]))
fig.update_layout(height=400, width=400, yaxis=dict(tickformat=".1%"))
fig.show()

df_plot = tour_data.groupby(['source','stop_frequency_in'])['tour_weight'].sum().reset_index()
df_plot['percentage'] = df_plot.groupby(['source'], group_keys=False)['tour_weight']. \
    apply(lambda x: x / float(x.sum()))

fig = px.bar(df_plot, x="stop_frequency_in", y="percentage", color="source",barmode="group",
             title="inbound stop frequency")
fig.for_each_annotation(lambda a: a.update(text = a.text.split("=")[-1]))
fig.update_layout(height=400, width=400, yaxis=dict(tickformat=".1%"))
fig.show()

In [7]:
tour_data= tour_data.assign(stop_frequency_total = lambda x: x['stop_frequency_out'] + x['stop_frequency_in'])

df_plot2 = tour_data.groupby(['source','stop_frequency_total'])['tour_weight'].sum().reset_index()
df_plot2['percentage'] = df_plot2.groupby(['source'], group_keys=False)['tour_weight']. \
    apply(lambda x: x / float(x.sum()))

fig = px.bar(df_plot2, x="stop_frequency_total", y="percentage", color="source",barmode="group",
             title="total stop frequency")
fig.for_each_annotation(lambda a: a.update(text = a.text.split("=")[-1]))
fig.update_layout(height=400, width=700, yaxis=dict(tickformat=".1%"), xaxis = dict(dtick = 1))
fig.show()

## stop frequency by tour type

In [8]:
def gen_stopfreq_by_tourtype(df: pd.DataFrame, in_out: str, type_value: str):
    df_plot = df.groupby(['source','tour_type',in_out])['tour_weight'].sum().reset_index()
    df_plot['percentage'] = df_plot.groupby(['source','tour_type'], group_keys=False)['tour_weight']. \
        apply(lambda x: x / float(x.sum()))
    df_plot['stop_freq_type'] = type_value
    return df_plot.rename(columns={in_out: 'frequency'})

df_plot = pd.concat([gen_stopfreq_by_tourtype(tour_data,'stop_frequency_out',"out"),
                     gen_stopfreq_by_tourtype(tour_data,'stop_frequency_in',"in"),
                     gen_stopfreq_by_tourtype(tour_data,'stop_frequency_total',"total")])

C:\Users\Modeller\AppData\Local\Temp\ipykernel_20796\3116939396.py:2: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\Modeller\AppData\Local\Temp\ipykernel_20796\3116939396.py:3: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\Modeller\AppData\Local\Temp\ipykernel_20796\3116939396.py:2: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\Modeller\AppData\Local\Temp\ipykernel_20796\3116939396.py:3: Future

In [9]:
def plot_stop_frequency(df: pd.DataFrame, plt_tour_type: str):

    df_plot_out = df.loc[(df['tour_type']==plt_tour_type) & (df['stop_freq_type']=="out")]
    df_plot_in = df.loc[(df['tour_type']==plt_tour_type) & (df['stop_freq_type']=="in")]
    df_plot_total = df.loc[(df['tour_type']==plt_tour_type) & (df['stop_freq_type']=="total")]

    fig = px.bar(df_plot_out,
                 x='frequency', y="percentage", color="source",barmode="group",
                 title=plt_tour_type + ": outbound stop frequency")
    fig.update_layout(height=350, width=350, yaxis=dict(tickformat=".1%"))
    fig.show()

    fig = px.bar(df_plot_in,
                 x='frequency', y="percentage", color="source",barmode="group",
                 title=plt_tour_type + ": intbound stop frequency")
    fig.update_layout(height=350, width=350, yaxis=dict(tickformat=".1%"))
    fig.show()

    fig = px.bar(df_plot_total,
                 x='frequency', y="percentage", color="source",barmode="group",
                 title=plt_tour_type + ": total stop frequency")
    fig.for_each_yaxis(lambda a: a.update(tickformat = ".1%"))
    fig.update_layout(height=350, width=700, yaxis=dict(tickformat=".1%"), xaxis = dict(dtick = 1))
    fig.show()

In [10]:
#| layout: [[1,1], [1]]
plot_stop_frequency(df_plot, "business")

In [11]:
#| layout: [[1,1], [1]]
plot_stop_frequency(df_plot, "eat")

In [12]:
#| layout: [[1,1], [1]]
plot_stop_frequency(df_plot, "eatout")

In [13]:
#| layout: [[1,1], [1]]
plot_stop_frequency(df_plot, "escort")

In [14]:
#| layout: [[1,1], [1]]
plot_stop_frequency(df_plot, "maint")

In [15]:
#| layout: [[1,1], [1]]
plot_stop_frequency(df_plot, "othdiscr")

In [16]:
#| layout: [[1,1], [1]]
plot_stop_frequency(df_plot, "othmaint")

In [17]:
#| layout: [[1,1], [1]]
plot_stop_frequency(df_plot, "school")

In [18]:
#| layout: [[1,1], [1]]
plot_stop_frequency(df_plot, "shopping")

In [19]:
#| layout: [[1,1], [1]]
plot_stop_frequency(df_plot, "social")

In [20]:
#| layout: [[1,1], [1]]
plot_stop_frequency(df_plot, "work")